# Extended Providers — native and OpenAI-compatible protocol validation

Extend a complete onboarding source with explicitly supplied AWS Bedrock, Bedrock Mantle,
native Gemini, Gemini OpenAI and Anthropic backend records. There are **no guessed provider
models**. Real IDs, endpoints, supportedModels and Key Vault references come from azd or explicit configuration;
model operations are inferred unless overridden by metadata or operation_overrides.

| Backend type | Protocol / Unified path | Backend credential format |
|---|---|---|
| aws-bedrock | Bedrock Converse /bedrock/model/{id}/converse | Full Bearer header |
| aws-bedrock-mantle | OpenAI /v1/chat/completions | Full Bearer header |
| gemini | generateContent /gemini/v1beta/models/{id}:generateContent | Raw x-goog-api-key |
| gemini-openai | OpenAI /v1/chat/completions | Full Bearer header |
| anthropic | Messages /claude/v1/messages | Raw x-api-key; separate anthropic-version |

Each backend requires its own named-value key and Key Vault secret URI. Native Gemini and
Gemini OpenAI must not reuse a secret containing the wrong header format. Inline secrets
are rejected, not saved or printed. Native payloads and SDK-independent HTTP calls remain visible.

**Shared routing side effect:** deployment regenerates global pools and policy fragments.
Run All uses the complete azd backend configuration, optional aliases/global settings, and SDK-discovered
APIM identity. Live backend coverage is checked before default-enabled deployment; source JSON is optional.
Shared backend routing is retained. Only a consumer access product newly created by this run is cleanup-eligible; reused products are retained.

## 1. Explicit supplied provider records and source completeness
Put a full onboarding parameter object in source_parameters_file. Supply new_backend_config records using real provider metadata; neither credentials nor model defaults are manufactured.

### 1. Explicit supplied provider records and source completeness

Configure this notebook's environment, explicit feature opt-ins, and run-owned resources without deploying anything.

**Input:** The selected repository kernel; edit the configuration values below and provide required secrets through environment variables.

**Output:** Configuration globals for the following initialization step; no cloud deployment or test is performed.


### Automatic models and optional overrides

Leave `environment=None`, azd enabled, `operation_overrides=None`, and
`skipped_models=None` to use the selected deployment without editing model lists.
Operations default to name-based **assumptions**, verified by real requests:
`image`/`flux` → image generation/edit; otherwise `embedding`/`text` → embeddings;
otherwise `gpt` → chat and Responses; other names → chat. Matching ignores case.

Optional examples (replace example names with names in the loaded inventory):

```python
# Replace operations only for these models; other models retain automatic mapping.
operation_overrides = {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
# Use operation_overrides = {} to discard stored azd operation overrides.
# Disable one model's tests with an empty override: {"my-model": []}.
skipped_models = ["mini", "legacy"]  # Case-insensitive substrings; backend deployment is unchanged.
overrides = {"resource_group": "rg-other-hub", "apim_name": "apim-other-hub"}  # Explicit azd-value overrides.
# To require a complete manually mapped catalogue: overrides["infer_operations"] = False.
```

Streaming is assumed for inferred chat models. Explicit mappings need `streaming`
when that scenario is selected. Responses GET/input-items returning HTTP 404 are warnings/SKIP for every model,
including explicit overrides, not successful retrieval. Other unsupported-operation
exceptions remain limited to inferred GPT-mini models; explicit mappings require
those non-404 retrieval checks. Authentication, throttling, and server failures
are never silently accepted.

### Scenario-specific input examples


`source_parameters_file` selects the complete existing backend contract.
Example: `source_parameters_file = root / "validation" / ".runs" / "backend-onboarding" / "your-run-id" / "backends" / "main.bicepparam"`.
Review it before setting `source_configuration_complete = True` and
`approve_shared_routing = True`. Provider additions require real endpoints and
credential references; they cannot be invented from a model name. For example:

```python
new_backend_config = [{
    "backendId": "my-gemini", "backendType": "gemini", "protocol": "gemini-native",
    "endpoint": "https://generativelanguage.googleapis.com",
    "authType": "api-key-gemini",
    "authConfig": {"namedValueKey": "my-provider-key",
                   "keyVaultSecretUri": "https://my-vault.vault.azure.net/secrets/provider-key"},
    "supportedModels": [{"name": "my-provider-model"}],
}]
```

Replace these illustrative names with provisioned resources. Native and
OpenAI-compatible endpoints require their own backend types and credentials.

Run All uses the complete selected azd backend inventory by default; full source JSON remains optional. Existing aliases/global settings are preserved and APIM identity is discovered by SDK. Deployment approval defaults to allow. Actual external provider endpoints and independent Key Vault credential references are still required for requested providers. Unavailable optional alias-model prerequisites and excluded test families produce SKIP, not a false PASS.


In [ ]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "azure.yaml").is_file() and (p / "shared" / "utils.py").is_file()), None)
if root is None:
    raise RuntimeError("Open this repository and run from its root or validation directory.")
sys.path.insert(0, str(root / "shared"))
import utils
with utils.notebook_step('1. Explicit supplied provider records and source completeness', inputs={'repository': str(repo_root if 'repo_root' in globals() else root)}, expected='Configuration globals for the following initialization step; no cloud deployment or test is performed.', namespace=globals()):
    from pathlib import Path
    import sys, json, time, uuid
    from urllib.parse import quote
    from requests.structures import CaseInsensitiveDict
    from validationtools import ValidationContext, ValidationRun, ModelCatalog, RequestTransport, ValidationBlocked
    from apimtools import APIMClientTool
    from contracttools import scoped_deployment_name
    from notebookdefaults import backend_source, source_catalog
    from contracttools import access_policy, prepare_artifact, deploy_contract
    from consumerhelpers import (
        access_artifact, contract_identifiers, cli_json, chat_content, error_code,
        complete_backend_parameters, load_contract_manifest, delete_owned_product, product_exists,
        review_backend_artifact,
    )
    environment = None
    from_azd = True
    overrides = {}  # Explicit canonical config/capabilities override the selected azd environment.
    run_id = time.strftime("%Y%m%d%H%M%S") + "-" + uuid.uuid4().hex[:6]
    inference_api_version = "2024-05-01-preview"
    openai_api_version = "2024-12-01-preview"
    cleanup_enabled = False

    source_parameters_file = None  # Optional authoritative complete-source override.
    source_configuration_complete = True
    approve_shared_routing = True
    new_backend_config = []  # Full new records; authConfig={namedValueKey, keyVaultSecretUri}; never secretValue.
    required_backend_types = ["aws-bedrock", "aws-bedrock-mantle", "gemini", "gemini-openai", "anthropic"]
    enable_foundry_weighted_alias = True
    enable_cross_cloud_openai_alias = True
    enable_anthropic_alias = True
    run = ValidationRun("extended-providers")
    run_dir = root / "validation" / ".runs" / "extended-providers" / run_id

    # None uses azd operation overrides when present, otherwise automatic name-based mapping.
    # Example: {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
    # An empty list for one model disables its tests; explicit lists replace that model's inferred operations.
    # A supplied dictionary replaces azd's operation map; {} resets it to automatic mapping.
    operation_overrides = None
    # None uses VALIDATION_SKIPPED_MODELS from azd (default []); matches substrings without case sensitivity.
    # Example: ["mini", "legacy"] skips any model containing either keyword in validation, not deployment.
    skipped_models = None


## 2. Lossless merge and capability catalog
Explicit operation annotations: chat for OpenAI-compatible, bedrock-converse, gemini-generate-content, anthropic-messages for native operations. Unknown support blocks enabled coverage.

### 2. Lossless merge and capability catalog

Resolve the selected deployment and validate this scenario's mandatory inputs before deployment or feature tests.

**Input:** Configuration from the previous cell, resource coordinates, inferred or explicitly overridden model operations, and enabled integration prerequisites.

**Output:** Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.


In [ ]:
with utils.notebook_step('2. Lossless merge and capability catalog', inputs={'required_backend_types': globals().get('required_backend_types'), 'enable_foundry_weighted_alias': globals().get('enable_foundry_weighted_alias'), 'enable_cross_cloud_openai_alias': globals().get('enable_cross_cloud_openai_alias'), 'enable_anthropic_alias': globals().get('enable_anthropic_alias'), 'source_parameters_file': globals().get('source_parameters_file'), 'new_backend_config': globals().get('new_backend_config')}, expected='Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.', namespace=globals()):
    context = ValidationContext.load(overrides=overrides, environment=environment, from_azd=from_azd,
        operation_overrides=operation_overrides, skipped_models=skipped_models)
    utils.require_inputs({f"gateway.{key}": context.config.get(key)
        for key in ("subscription_id", "resource_group", "location", "apim_name")})
    apim = APIMClientTool(context.config["resource_group"], context.config["apim_name"])
    apim.initialize_context(context.account, enumerate_subscriptions=False)
    source_parameters = backend_source(context, apim, source_parameters_file,
        overrides.get("backend_template_options"))
    backend_parameters = complete_backend_parameters(source_parameters, new_backend_config,
        confirmed_complete=source_configuration_complete)
    context.catalog = source_catalog(context, backend_parameters)
    transport = RequestTransport(timeout=60)
    operations = {
        "ai-foundry": "chat", "azure-openai": "chat",
        "aws-bedrock-mantle": "chat", "gemini-openai": "chat",
        "aws-bedrock": "bedrock-converse", "gemini": "gemini-generate-content",
        "anthropic": "anthropic-messages",
    }
    provider_models = {}
    configured_types = {backend["backendType"] for backend in backend_parameters["llmBackendConfig"]}
    utils.require_inputs({f"external.{backend_type}.backend_endpoint_and_credentials":
        True if backend_type in configured_types else None for backend_type in required_backend_types})
    for backend_type, operation in operations.items():
        if backend_type not in required_backend_types and backend_type not in ("ai-foundry", "azure-openai"):
            provider_models[backend_type] = []
            run.skip(f"{backend_type}/coverage", "Provider excluded explicitly.")
            continue
        eligible = context.catalog.models_for(operation, backend_type=backend_type)
        # A default inferred chat model uses its provider's native chat wire format.
        # Explicit operation lists remain authoritative, including an empty list.
        if operation != "chat":
            eligible = list(dict.fromkeys(eligible + [
                entry.name for entry in context.catalog.entries
                if entry.backend_type == backend_type and entry.operations_source == "inferred"
                and entry.name in context.catalog.models_for("chat", backend_type=backend_type)
            ]))
        provider_models[backend_type] = eligible
        if not eligible:
            run.skip(f"{backend_type}/models", "No eligible model supports this provider operation after operation/keyword filtering.")
    auth_types = {"aws-bedrock": "api-key-bearer", "aws-bedrock-mantle": "api-key-bearer",
        "gemini": "api-key-gemini", "gemini-openai": "api-key-bearer", "anthropic": "api-key-anthropic"}
    seen_named_values = set()
    credential_formats = {}
    for backend in backend_parameters["llmBackendConfig"]:
        if backend["backendType"] not in auth_types:
            continue
        auth = backend.get("authConfig") or {}
        utils.require_inputs({f"external.{backend['backendId']}.endpoint": backend.get("endpoint"),
            f"external.{backend['backendId']}.namedValueKey": auth.get("namedValueKey"),
            f"external.{backend['backendId']}.keyVaultSecretUri": auth.get("keyVaultSecretUri")})
        if backend.get("authType") != auth_types[backend["backendType"]]:
            raise ValueError(f"Wrong header credential type for backend {backend['backendId']}.")
        if auth["namedValueKey"] in seen_named_values:
            raise ValueError("Provider backends require independent named values.")
        seen_named_values.add(auth["namedValueKey"])
        previous_format = credential_formats.get(auth["keyVaultSecretUri"])
        if previous_format and previous_format != backend["authType"]:
            raise ValueError("Do not share a provider secret across raw-key and bearer formats.")
        credential_formats[auth["keyVaultSecretUri"]] = backend["authType"]
    utils.format_output("Provider initialization", {"provider_models": provider_models,
        "deployment_backends": len(backend_parameters["llmBackendConfig"])}, status="info", part="RESULT")
    run_dir.mkdir(parents=True, exist_ok=True)


## 3. Same-wire-protocol aliases — feature definitions remain visible
Never mix Converse, generateContent and Anthropic Messages under an OpenAI alias. Cross-cloud members must explicitly declare the same nonempty protocol (for example, openai-compat) in source metadata. A native Anthropic alias is Anthropic-only today, not a claimed cross-cloud translation layer.

### 3. Same-wire-protocol aliases — feature definitions remain visible

Prepare the 3. Same-wire-protocol aliases — feature definitions remain visible state shown below for its subsequent feature-specific requests and assertions.

**Input:** `enable_foundry_weighted_alias`, `enable_cross_cloud_openai_alias`, `enable_anthropic_alias`, `backend_parameters`, `provider_models`

**Output:** `new_aliases`, `members`, `exc`, `existing_aliases` are available to the following cells.


In [ ]:
with utils.notebook_step('3. Same-wire-protocol aliases — feature definitions remain visible', inputs={'enable_foundry_weighted_alias': globals().get('enable_foundry_weighted_alias'), 'enable_cross_cloud_openai_alias': globals().get('enable_cross_cloud_openai_alias'), 'enable_anthropic_alias': globals().get('enable_anthropic_alias'), 'backend_parameters': globals().get('backend_parameters'), 'provider_models': globals().get('provider_models')}, expected='`new_aliases`, `members`, `exc`, `existing_aliases` are available to the following cells.', namespace=globals()):
    new_aliases = []
    if enable_foundry_weighted_alias:
        members = provider_models.get("ai-foundry", [])[:3]
        if len(members) < 2:
            run.skip("alias/foundry-weighted", "At least two chat-capable Foundry models are required.")
        else:
            try:
                members = context.catalog.alias_members(members, "chat", backend_type="ai-foundry", min_members=2)
            except ValidationBlocked as exc:
                run.skip("alias/foundry-weighted", str(exc))
                members = []
            if members:
                new_aliases.append({"name": "validation-providers-foundry-weighted", "models": members, "strategy": "weighted",
                                   "weights": [60, 40] if len(members) == 2 else [50, 30, 20]})
    else:
        run.skip("alias/foundry-weighted", "Disabled explicitly.")
    if enable_cross_cloud_openai_alias:
        members = [provider_models[backend_type][0] for backend_type in ("ai-foundry", "aws-bedrock-mantle", "gemini-openai")
                   if provider_models.get(backend_type)]
        members = list(dict.fromkeys(members))
        if len(members) < 2:
            run.skip("alias/cross-cloud-openai", "Two distinct models across compatible providers are required.")
        else:
            try:
                members = context.catalog.alias_members(members, "chat", min_members=2)
            except ValidationBlocked as exc:
                run.skip("alias/cross-cloud-openai", f"Declare a common wire protocol for all members: {exc}")
                members = []
            if members:
                new_aliases.append({"name": "validation-providers-cross-cloud-openai", "models": members, "strategy": "priority"})
    else:
        run.skip("alias/cross-cloud-openai", "Disabled explicitly.")
    if enable_anthropic_alias:
        members = provider_models.get("anthropic", [])
        if not members:
            run.skip("alias/anthropic", "At least one annotated Anthropic Messages model is required.")
        else:
            if all("anthropic-messages" in context.catalog.operations(model) for model in members[:3]):
                try:
                    members = context.catalog.alias_members(members[:3], "anthropic-messages", backend_type="anthropic")
                except ValidationBlocked as exc:
                    run.skip("alias/anthropic", str(exc))
                    members = []
            else:
                run.skip("alias/anthropic", "Direct native tests use inferred chat; a native alias requires explicit native operation metadata.")
                members = []
            if members:
                new_aliases.append({"name": "validation-providers-anthropic-messages", "models": members[:3], "strategy": "priority"})
    else:
        run.skip("alias/anthropic", "Disabled explicitly.")
    aliases_by_name = {alias["name"]: alias for alias in backend_parameters["modelAliases"]}
    aliases_by_name.update((alias["name"], alias) for alias in new_aliases)
    backend_parameters["modelAliases"] = list(aliases_by_name.values())
    utils.format_output('3. Same-wire-protocol aliases — feature definitions remain visible', {'value_1': "New same-protocol aliases:", 'new_aliases': new_aliases}, status='info', part='RESULT')
    providers_ready = any(provider_models.values())
    if not providers_ready:
        run.skip("provider-tests", "No eligible provider models remain; no access/test deployment is performed.")


## 4. Before inventory, complete-source guard and editable parameter preview
All live backend IDs must be retained. Review the full effective parameters and side effects before approval.

### 4. Before inventory, complete-source guard and editable parameter preview

Generate the editable parameter artifact for 4. Before inventory, complete-source guard and editable parameter preview and display the proposed configuration without deploying it.

**Input:** `backend_parameters`, `backend`, `run_dir`

**Output:** A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.


In [ ]:
with utils.notebook_step('4. Before inventory, complete-source guard and editable parameter preview', inputs={'backend_parameters': globals().get('backend_parameters'), 'backend': globals().get('backend'), 'run_dir': globals().get('run_dir')}, expected='A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.', namespace=globals()):
    before_backends, before_pools = apim.get_backends()
    expected_backend_ids = {backend["backendId"] for backend in backend_parameters["llmBackendConfig"]}
    omitted = {backend["name"] for backend in before_backends} - expected_backend_ids
    if omitted:
        run.blocked("shared-source-inventory", f"Full source omitted live backends: {sorted(omitted)}")
        raise RuntimeError("Unsafe partial source.")
    backend_parameters["apim"] = {
        "subscriptionId": context.config["subscription_id"],
        "resourceGroupName": context.config["resource_group"], "name": apim.apim_resource_name,
    }
    if not backend_parameters.get("apimManagedIdentity"):
        raise ValueError("Complete source must include apimManagedIdentity coordinates.")
    backend_artifact = prepare_artifact(root / "bicep" / "infra" / "llm-backend-onboarding" / "main.bicep",
                                        backend_parameters, run_dir / "backends")
    utils.format_output('4. Before inventory, complete-source guard and editable parameter preview', {'value_1': "Before backend IDs:", 'value_2': [backend["name"] for backend in before_backends]}, status='info', part='RESULT')
    utils.format_output('4. Before inventory, complete-source guard and editable parameter preview', {'value_1': "Before pool names:", 'value_2': [pool["name"] for pool in before_pools]}, status='info', part='RESULT')
    utils.format_output('4. Before inventory, complete-source guard and editable parameter preview', {'value_1': backend_artifact.preview()}, status='info', part='RESULT')
    utils.format_output('4. Before inventory, complete-source guard and editable parameter preview', {'value_1': "Primary editable input:", 'value_2': backend_artifact.params_file}, status='info', part='RESULT')


## 5. Approved routing deployment and independent after-state inspection
Output-reported pools are reconciled with live APIM; outputs alone cannot validate routing.

### 5. Approved routing deployment and independent after-state inspection

Deploy the reviewed artifact for 5. Approved routing deployment and independent after-state inspection, then inspect its real resource outputs and ownership.

**Input:** `approve_shared_routing`, `backend_artifact`, `source_parameters`, `backend_parameters`, `backend`

**Output:** Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.


In [ ]:
with utils.notebook_step('5. Approved routing deployment and independent after-state inspection', inputs={'approve_shared_routing': globals().get('approve_shared_routing'), 'backend_artifact': globals().get('backend_artifact'), 'source_parameters': globals().get('source_parameters'), 'backend_parameters': globals().get('backend_parameters'), 'backend': globals().get('backend')}, expected='Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.', namespace=globals()):
    if not approve_shared_routing:
        run.blocked("shared-routing-deploy", "Review the complete .bicepparam; set approve_shared_routing=True.")
        raise RuntimeError("Shared routing approval is required.")
    effective = review_backend_artifact(backend_artifact, source_parameters, backend_parameters["apim"],
                                        {item["name"] for item in before_backends},
                                        editable_aliases={alias["name"] for alias in new_aliases})
    expected_backend_ids = {item["backendId"] for item in effective["llmBackendConfig"]}
    backend_outputs = deploy_contract(backend_artifact.params_file, context.config["location"], scoped_deployment_name("validation-providers", apim.apim_service_id))
    after_backends, after_pools = apim.get_backends()
    live_backend_ids = {backend["name"] for backend in after_backends}
    live_pool_names = {pool["name"] for pool in after_pools}
    expected_pools = set(backend_outputs["poolNames"]["value"])
    run.check("onboarding/backends", expected_backend_ids <= live_backend_ids,
              expected=sorted(expected_backend_ids), actual=sorted(live_backend_ids))
    run.check("onboarding/pools", bool(expected_pools) and expected_pools <= live_pool_names,
              expected=sorted(expected_pools), actual=sorted(live_pool_names))
    for fragment_name in backend_outputs["policyFragments"]["value"].values():
        if isinstance(fragment_name, str):
            fragment = apim.client.policy_fragment.get(
                context.config["resource_group"], apim.apim_resource_name, fragment_name,
            )
            run.check(f"onboarding/fragment/{fragment_name}", bool(fragment.value),
                      expected="Live deployed policy fragment", actual=fragment_name)
    models_api = apim.discover_api_exact("models")
    unified_api = apim.discover_api_exact("unified-ai")


## 6. Configure and preview a stable access contract
Grant only the selected provider models and aliases; capacity/debug policy is visible. The contract covers Universal and Unified APIs.

### 6. Configure and preview a stable access contract

Prepare the stable access contract state shown below for its subsequent feature-specific requests and assertions.

**Input:** `run_dir`, `run_id`, `alias`, `new_aliases`

**Output:** `allowed_models`, `use_case`, `product_id`, `policy_xml`, `access` are available to the following cells.


In [ ]:
with utils.notebook_step('6. Configure and preview a stable access contract', inputs={'run_dir': globals().get('run_dir'), 'run_id': globals().get('run_id'), 'alias': globals().get('alias'), 'new_aliases': globals().get('new_aliases')}, expected='`allowed_models`, `use_case`, `product_id`, `policy_xml`, `access` are available to the following cells.', namespace=globals()):
    if not providers_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if providers_ready:
        allowed_models = list(dict.fromkeys([model for models in provider_models.values() for model in models]
                                          + [alias["name"] for alias in new_aliases]))
        use_case = {"businessUnit": "Validation", "useCaseName": "Providers", "environment": "DEV"}
        product_id = f"LLM-{use_case['businessUnit']}-{use_case['useCaseName']}-{use_case['environment']}"
        policy_xml = access_policy(allowed_models, 10000, 100000, "Daily",
                                  inbound_extra='<set-variable name="enableResponseHeaders" value="@(true)" />')
        utils.format_output('6. Configure and preview a stable access contract', {'policy_xml': policy_xml}, status='info', part='RESULT')
        access = access_artifact(root, run_dir / "access", context, apim, use_case,
                                 ["universal-llm-api", "unified-ai-api"], policy_xml)
        utils.format_output('6. Configure and preview a stable access contract', {'value_1': access.preview()}, status='info', part='RESULT')


## 7. Separate access deployment and exact subscription credential

### 7. Separate access deployment and exact subscription credential

Deploy the reviewed artifact for 7. Separate access deployment and exact subscription credential, then inspect its real resource outputs and ownership.

**Input:** `product_id`, `access`, `models_api`, `unified_api`

**Output:** Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.


In [ ]:
with utils.notebook_step('7. Separate access deployment and exact subscription credential', inputs={'product_id': globals().get('product_id'), 'access': globals().get('access'), 'models_api': globals().get('models_api'), 'unified_api': globals().get('unified_api')}, expected='Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.', namespace=globals()):
    if not providers_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if providers_ready:
        product_preexists = product_exists(context, apim, product_id)
        outputs = deploy_contract(access.params_file, context.config["location"], scoped_deployment_name("validation-providers-access", apim.apim_service_id))
        product_id, subscription_name = contract_identifiers(outputs, product_id)
        api_key = apim.get_subscription_key(subscription_name)
        access.record_resource(f"{apim.apim_service_id}/products/{product_id}", created=not product_preexists)
        headers = {"api-key": api_key}

        def record(case, result, expected_status=200, body_check=None, **evidence):
            result.headers = CaseInsensitiveDict(result.headers)
            valid_body = True
            if body_check is not None:
                try:
                    valid_body = bool(body_check(result.json_body))
                except (TypeError, ValueError, KeyError, IndexError, AttributeError):
                    valid_body = False
            run.check(
                case, not result.error and result.status_code == expected_status and valid_body,
                expected=f"HTTP {expected_status}; expected response shape/policy evidence",
                actual=result.error or str(result.status_code),
                duration=result.duration,
                headers={k: v for k, v in result.headers.items() if k.lower().startswith("uaig-")},
                **evidence,
            )
            return result

        for label, descriptor in [("universal", models_api), ("unified", unified_api)]:
            record(f"{label}/discovery", transport.request("GET", f"{descriptor['url']}/deployments", headers=headers),
                   body_check=lambda data: {item["name"] for item in data["value"]} == set(allowed_models))


## 8. OpenAI-compatible chat on BOTH API surfaces
Only explicitly compatible backend types participate. A routed provider 400 is still a failure, not success.

### 8. OpenAI-compatible chat on BOTH API surfaces

Exercise 8. OpenAI-compatible chat on BOTH API surfaces and compare the actual response/evidence with the expected feature behavior.

**Input:** `models`, `backend_type`, `provider_models`, `inference_api_version`, `model`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('8. OpenAI-compatible chat on BOTH API surfaces', inputs={'models': globals().get('models'), 'backend_type': globals().get('backend_type'), 'provider_models': globals().get('provider_models'), 'inference_api_version': globals().get('inference_api_version'), 'model': globals().get('model')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    if not providers_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if providers_ready:
        messages = [{"role": "user", "content": "Reply with the single word OK."}]
        for backend_type in ("ai-foundry", "azure-openai", "aws-bedrock-mantle", "gemini-openai"):
            models = provider_models.get(backend_type, [])
            if not models:
                run.skip(f"{backend_type}/openai-chat", "No optional compatible models configured.")
            for model in models:
                for surface, url in [
                    ("universal", f"{models_api['url']}/chat/completions?api-version={inference_api_version}"),
                    ("unified", f"{unified_api['url']}/v1/chat/completions"),
                ]:
                    result = record(f"{backend_type}/{surface}/{model}",
                                    transport.request("POST", url, headers=headers, json={"model": model, "messages": messages}),
                                    body_check=lambda data: bool(chat_content(data)))
                    run.check(f"{backend_type}/{surface}/{model}/routing",
                              result.headers.get("UAIG-Model-Id") == model and bool(result.headers.get("UAIG-Backend")),
                              expected="Requested model and observable backend", actual=result.headers.get("UAIG-Model-Id"))


## 9. Native Bedrock Converse — visible native body and escaped model path
Response must contain nonempty output.message.content text and bedrock-native routing evidence.

### 9. Native Bedrock Converse — visible native body and escaped model path

Exercise 9. Native Bedrock Converse — visible native body and escaped model path and compare the actual response/evidence with the expected feature behavior.

**Input:** `provider_models`, `url`, `unified_api`, `model`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('9. Native Bedrock Converse — visible native body and escaped model path', inputs={'provider_models': globals().get('provider_models'), 'url': globals().get('url'), 'unified_api': globals().get('unified_api'), 'model': globals().get('model')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    if not providers_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if providers_ready:
        for model in provider_models.get("aws-bedrock", []):
            url = f"{unified_api['url']}/bedrock/model/{quote(model, safe='')}/converse"
            body = {"messages": [{"role": "user", "content": [{"text": "Reply with the single word OK."}]}],
                    "inferenceConfig": {"maxTokens": 32, "temperature": 0.0}}
            result = record(f"bedrock/converse/{model}", transport.request("POST", url, headers=headers, json=body),
                            body_check=lambda data: any(block.get("text", "").strip()
                                for block in data.get("output", {}).get("message", {}).get("content", [])))
            run.check(f"bedrock/routing/{model}", result.headers.get("UAIG-API-Type") == "bedrock-native"
                      and result.headers.get("UAIG-Model-Id") == model,
                      expected="bedrock-native; requested model", actual=result.headers.get("UAIG-API-Type"))


## 10. Native Gemini generateContent — raw key backend, native schema

### 10. Native Gemini generateContent — raw key backend, native schema

Exercise 10. Native Gemini generateContent — raw key backend, native schema and compare the actual response/evidence with the expected feature behavior.

**Input:** `provider_models`, `url`, `unified_api`, `model`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('10. Native Gemini generateContent — raw key backend, native schema', inputs={'provider_models': globals().get('provider_models'), 'url': globals().get('url'), 'unified_api': globals().get('unified_api'), 'model': globals().get('model')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    if not providers_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if providers_ready:
        for model in provider_models.get("gemini", []):
            url = f"{unified_api['url']}/gemini/v1beta/models/{quote(model, safe='')}:generateContent"
            body = {"contents": [{"role": "user", "parts": [{"text": "Reply with the single word OK."}]}],
                    "generationConfig": {"maxOutputTokens": 32, "temperature": 0.0}}
            result = record(f"gemini/generateContent/{model}", transport.request("POST", url, headers=headers, json=body),
                            body_check=lambda data: any(part.get("text", "").strip()
                                for candidate in data.get("candidates", []) for part in candidate.get("content", {}).get("parts", [])))
            run.check(f"gemini/routing/{model}", result.headers.get("UAIG-API-Type") == "gemini-native"
                      and result.headers.get("UAIG-Model-Id") == model,
                      expected="gemini-native; requested model", actual=result.headers.get("UAIG-API-Type"))


## 11. Native Anthropic Messages — no Converse/OpenAI translation
The source anthropicVersion option controls the separate backend header.

### 11. Native Anthropic Messages — no Converse/OpenAI translation

Exercise 11. Native Anthropic Messages — no Converse/OpenAI translation and compare the actual response/evidence with the expected feature behavior.

**Input:** `provider_models`, `model`, `headers`, `body`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('11. Native Anthropic Messages — no Converse/OpenAI translation', inputs={'provider_models': globals().get('provider_models'), 'model': globals().get('model'), 'headers': globals().get('headers'), 'body': globals().get('body')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    if not providers_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if providers_ready:
        for model in provider_models.get("anthropic", []):
            body = {"model": model, "max_tokens": 32, "messages": [{"role": "user", "content": "Reply with OK."}]}
            result = record(f"anthropic/messages/{model}",
                            transport.request("POST", f"{unified_api['url']}/claude/v1/messages", headers=headers, json=body),
                            body_check=lambda data: any(block.get("text", "").strip() for block in data.get("content", [])
                                                        if block.get("type") == "text"))
            run.check(f"anthropic/routing/{model}", result.headers.get("UAIG-API-Type") == "claude-native"
                      and result.headers.get("UAIG-Model-Id") == model,
                      expected="claude-native; requested model", actual=result.headers.get("UAIG-API-Type"))


## 12. Same-protocol alias consumption
Assert alias identity and resolved membership; incompatible-member rejections fail rather than passing. Native Anthropic aliases use Messages only.

### 12. Same-protocol alias consumption

Exercise 12. Same-protocol alias consumption and compare the actual response/evidence with the expected feature behavior.

**Input:** `new_aliases`, `messages`, `alias`, `url`, `valid_body`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('12. Same-protocol alias consumption', inputs={'new_aliases': globals().get('new_aliases'), 'messages': globals().get('messages'), 'alias': globals().get('alias'), 'url': globals().get('url'), 'valid_body': globals().get('valid_body')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    if not providers_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if providers_ready:
        for alias in new_aliases:
            if alias["name"].startswith("anthropic-messages-"):
                surfaces = [("claude-native", f"{unified_api['url']}/claude/v1/messages")]
                body = {"model": alias["name"], "max_tokens": 32, "messages": messages}
                valid_body = lambda data: any(block.get("text", "").strip() for block in data.get("content", [])
                                              if block.get("type") == "text")
            else:
                surfaces = [("universal", f"{models_api['url']}/chat/completions?api-version={inference_api_version}"),
                            ("openai-compat", f"{unified_api['url']}/v1/chat/completions")]
                body = {"model": alias["name"], "messages": messages}
                valid_body = lambda data: bool(chat_content(data))
            for surface, url in surfaces:
                result = record(f"aliases/{alias['name']}/{surface}",
                                transport.request("POST", url, headers=headers, json=body), body_check=valid_body)
                run.check(f"aliases/{alias['name']}/{surface}/membership",
                          result.headers.get("UAIG-Alias") == alias["name"]
                          and result.headers.get("UAIG-Resolved-Model") in alias["models"],
                          expected={"alias": alias["name"], "members": alias["models"]},
                          actual={key: result.headers.get(key) for key in ("UAIG-Alias", "UAIG-Resolved-Model")})


## 13. Compatible-pool filtering — native-only models cannot leak to the OpenAI surface
When a name intentionally appears in both native and compatible pools, positive routing checks apply instead of assuming rejection.

### 13. Compatible-pool filtering — native-only models cannot leak to the OpenAI surface

Collect the final 13. Compatible-pool filtering — native-only models cannot leak to the OpenAI surface verdict without discarding earlier failed or blocked required cases.

**Input:** `model`, `backend_type`, `provider_models`, `headers`

**Output:** The strict validation summary; any failed or blocked required coverage raises.


In [ ]:
with utils.notebook_step('13. Compatible-pool filtering — native-only models cannot leak to the OpenAI surface', inputs={'model': globals().get('model'), 'backend_type': globals().get('backend_type'), 'provider_models': globals().get('provider_models'), 'headers': globals().get('headers')}, expected='The strict validation summary; any failed or blocked required coverage raises.', namespace=globals()):
    if not providers_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if providers_ready:
        compatible_models = {model for backend_type in ("ai-foundry", "azure-openai", "aws-bedrock-mantle", "gemini-openai")
                             for model in provider_models.get(backend_type, [])}
        for backend_type in ("aws-bedrock", "gemini", "anthropic"):
            for model in provider_models.get(backend_type, []):
                if model in compatible_models:
                    run.skip(f"pool-filter/{backend_type}/{model}", "Same name has an explicitly compatible pool; positive routing covered.")
                    continue
                record(f"pool-filter/{backend_type}/{model}",
                       transport.request("POST", f"{unified_api['url']}/v1/chat/completions", headers=headers,
                                         json={"model": model, "messages": messages}), 400,
                       body_check=lambda data: error_code(data) == "model_not_available_on_this_surface")
        run.finish()


## 14. Opt-in owned access cleanup
Do not delete provider backends, their secret references, pools or shared aliases. Retained configuration and generated artifacts remain available for review.

### 14. Opt-in owned access cleanup

Apply the explicitly enabled 14. Opt-in owned access cleanup action only to resources owned by this run.

**Input:** `cleanup_enabled`, `product_id`, `access`

**Output:** Ownership-scoped deletion results, or an explicit disabled-cleanup message; predecessor resources are preserved.


In [ ]:
with utils.notebook_step('14. Opt-in owned access cleanup', inputs={'cleanup_enabled': globals().get('cleanup_enabled'), 'product_id': globals().get('product_id'), 'access': globals().get('access')}, expected='Ownership-scoped deletion results, or an explicit disabled-cleanup message; predecessor resources are preserved.', namespace=globals()):
    if not providers_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if providers_ready:
        utils.format_output('14. Opt-in owned access cleanup', {'value_1': "Owned access cleanup candidates:", 'value_2': access.cleanup_candidates()}, status='info', part='RESULT')
        if cleanup_enabled:
            delete_owned_product(context, apim, product_id, artifact=access)
        else:
            utils.format_output('14. Opt-in owned access cleanup', {'value_1': "Cleanup disabled; shared routing is retained."}, status='info', part='RESULT')
    if not providers_ready:
        utils.format_output("Provider scenario", {"reason": "No eligible provider models; no inference tests executed."}, status="SKIP", part="RESULT")
